# Image Classification Workshop Lab: RGB Variant

This notebook is designed as a first hands-on image-classification lesson for beginners.

This RGB variant keeps the workflow aligned across models by giving both the classical baselines and the CNN color images.

We will:
- load an image dataset from folders,
- choose a classification target,
- inspect class balance and image examples,
- train a simple scikit-learn baseline on RGB inputs,
- train a tiny CNN on the same RGB inputs when PyTorch is available,
- compare accuracy, runtime, and workflow tradeoffs,
- stress-test device predictions on damaged images from the `Defect` folder.

The notebook supports two targets:
- `device_type`: classify `device1`, `device 2`, `device 3`, and future device folders using only the clean device folders for training.
- `damage_status`: classify `Damaged` vs `Not-Damaged` by treating `Defect` as damaged and the normal device folders as not damaged.


## 1. Imports and workshop helpers

This cell loads the tools we need and prepares a runtime log so we can measure how long each major step takes.

In [ ]:
# Standard library tools for file paths, timing, and warning control.
from pathlib import Path
import math
import sys
import time
import warnings

# Data-science libraries for plotting, arrays, tables, and image loading.
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from PIL import Image
from IPython.display import Markdown, display
# Classical machine-learning models and helpers used later in the notebook.
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier

# Find the notebook folder so imports and dataset paths work even if the user runs
# the notebook from the repository root instead of inside 0_Class_Notebooks.
NOTEBOOK_DIR = Path.cwd()
if NOTEBOOK_DIR.name != "0_Class_Notebooks":
    candidate = NOTEBOOK_DIR / "0_Class_Notebooks"
    if candidate.exists():
        NOTEBOOK_DIR = candidate

# Add the notebook folder to Python's import path so our helper file can be imported.
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from workshop_image_utils import (
    available_target_modes,
    create_progress,
    load_image_records,
    load_images_as_arrays,
    record_timing,
    safe_stratify_labels,
    target_column_for_mode,
    timing_frame,
    update_progress,
    validate_target_mode,
)

# Keep charts readable and reduce distracting warning messages for beginners.
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)

# PyTorch is optional in this workshop, so we detect whether it is installed.
try:
    import torch
    from torch import nn
    from torch.utils.data import DataLoader, TensorDataset
    TORCH_AVAILABLE = True
except ImportError:
    TORCH_AVAILABLE = False
    torch = None
    nn = None
    DataLoader = None
    TensorDataset = None
    print("PyTorch is not installed in this environment, so the CNN section will be skipped.")

# We will store timing measurements here as we move through the notebook.
step_timings = []


## 2. Configuration

Change the values in this cell when you want to switch datasets or tasks.

Beginner note:
- `device_type` trains on clean device folders and can use the `Defect` folder as a separate challenge set.
- `damage_status` asks a binary question: damaged or not damaged.
- Some values in the configuration cell are also hyperparameters, which means they control model behavior rather than just file paths.

Quick guide to the shared settings:
- `IMAGE_SIZE`: resize target for the classical image models. In this RGB notebook, each image contributes three color channels, so larger values create many more input features.
- `TEST_SIZE`: fraction of the dataset held out for testing.
- `RANDOM_STATE`: keeps the split reproducible so students see the same result again.
- `USE_PCA`: turns dimensionality reduction on or off for the scaled flattened-pixel models.
- `PCA_VARIANCE_TO_KEEP`: when PCA is on, keep this share of the original pixel variation.
- `CNN_IMAGE_SIZE`: resize target for the CNN.
- `CNN_EPOCHS`: number of full passes through the training set.
- `CNN_BATCH_SIZE`: how many images the CNN processes before each weight update.
- `CNN_LEARNING_RATE`: how large each CNN optimization step is.

### Student change points

Change one value at a time, rerun from this cell downward, and compare the accuracy, confusion matrix, wrong predictions, and runtime.

- Start with `TARGET_MODE`, `IMAGE_SIZE`, and `TEST_SIZE`.
- Compare this notebook with the grayscale notebook: did RGB color help, hurt, or mostly add extra computation?
- Then try `USE_PCA`, model settings such as `KNeighborsClassifier(n_neighbors=...)`, `DecisionTreeClassifier(max_depth=...)`, or `CNN_EPOCHS`.
- Inspect mistakes before deciding which model is best.


In [ ]:
DATASET_ROOT = (NOTEBOOK_DIR.parent / "Augmented_Device_Images").resolve()
METADATA_PATH = DATASET_ROOT / "metadata.csv"  # Optional future file.

################## Classification Mode ##################################
TARGET_MODE = "device_type"  # Student change point: try "device_type" and compare the task difficulty.
INCLUDED_DEVICE_TYPES = ["device1", "device 2"]  # Student change point: add/remove available device folders.

########## Shared Hyperparameters ##################################
IMAGE_SIZE = 96  # Student change point: RGB has 3 channels, so larger sizes get expensive quickly.
TEST_SIZE = 0.30  # Student change point: try 0.20 or 0.40 and inspect whether scores change.
USE_PCA = True  # Student change point: turn on PCA for the scaled flattened-pixel models.
PCA_VARIANCE_TO_KEEP = 0.95  # Student change point: try 0.90, 0.95, or 0.99 when USE_PCA is True.

########### CNN Hyperparameters ##################################
CNN_IMAGE_SIZE = 64  # Student change point: controls image size for the CNN only.
CNN_EPOCHS = 6  # Student change point: more epochs may improve learning or overfit.
CNN_BATCH_SIZE = 8  # Student change point: changes how many images the CNN sees per update.
CNN_LEARNING_RATE = 0.001  # Student change point: too high or too low can hurt learning.

RANDOM_STATE = 42  # Student change point: change this to see how a different split affects results.
RUN_DEFECT_CHALLENGE = TARGET_MODE == "device_type"
print(f"Dataset root: {DATASET_ROOT}")
print(f"Metadata file exists: {METADATA_PATH.exists()}")
print(f"Current target mode: {TARGET_MODE}")
print(f"Included device folders: {INCLUDED_DEVICE_TYPES}")
print(f"Run defect challenge after device training: {RUN_DEFECT_CHALLENGE}")
print(f"PCA for scaled flattened-pixel models: {USE_PCA} (variance kept: {PCA_VARIANCE_TO_KEEP:.0%})")


## 3. Load the dataset and inspect available labels

Stop and predict before you run this cell:
- Which task do you think will be easier for a model: device type or damage status?
- What problems could appear if one class has many more images than another?
- Do you think a model trained on clean devices will still recognize damaged ones?


In [ ]:
# Load the images into memory and resize them to a shared shape so every model
# starts from the same visual input.
started = time.perf_counter()
all_records = load_image_records(DATASET_ROOT, metadata_path=METADATA_PATH if METADATA_PATH.exists() else None)
load_elapsed = record_timing(step_timings, "dataset_scan", started)

print(f"Loaded {len(all_records)} image records in {load_elapsed:.2f} seconds before workshop filtering.")

standard_device_records = all_records[all_records["is_standard_device_image"]].reset_index(drop=True)
standard_device_records = standard_device_records[standard_device_records["device_type"].isin(INCLUDED_DEVICE_TYPES)].reset_index(drop=True)
defect_challenge_records = all_records[all_records["is_defect_challenge"]].reset_index(drop=True)

print(f"Standard device records after included-device filtering: {len(standard_device_records)}")
print(f"Defect challenge records available: {len(defect_challenge_records)}")

if TARGET_MODE == "device_type":
    records = standard_device_records.copy()
else:
    records = pd.concat([standard_device_records, defect_challenge_records], ignore_index=True)

display(records.head())
print("Available target modes right now:", available_target_modes(records))

is_valid_target, target_message = validate_target_mode(records, TARGET_MODE)
print(target_message)
if not is_valid_target:
    raise ValueError(target_message)

target_column = target_column_for_mode(TARGET_MODE)
records = records[records[target_column].notna()].reset_index(drop=True)
print(f"Records remaining after filtering to labeled rows for {TARGET_MODE}: {len(records)}")

if TARGET_MODE == "device_type" and RUN_DEFECT_CHALLENGE:
    print("Defect images will stay outside the train/test split and be used later as a challenge set.")


## 4. Class balance and image inventory

This section helps us answer:
- How many examples do we have for each class?
- Is the dataset balanced?
- Are we at risk of teaching the model from a tiny or biased sample?

For `device_type`, this chart only shows the clean training pool. The damaged `Defect` images are held back for the later stress test.
For `damage_status`, the chart can include both original and augmented clean device images because they all act as `Not-Damaged` examples.


In [ ]:
class_counts = records[target_column].value_counts(dropna=True).sort_index()
display(class_counts.rename("count").to_frame())

ax = class_counts.plot(kind="bar", color="#1f77b4", title=f"Class counts for {TARGET_MODE}")
ax.set_xlabel("Class label")
ax.set_ylabel("Number of images")
plt.xticks(rotation=20)
plt.show()

if TARGET_MODE == "damage_status":
    print("Damage-status note:")
    print("Both original and augmented clean device images count as `Not-Damaged` in this binary task.")
    print("Defect images and their augmentations count as `Damaged`.")

print("Reflection prompt:")
print("If one class dominates this chart, the model may learn shortcuts instead of the real visual concept.")


## 5. Preview example images

Seeing the data matters. Beginners should always inspect a few images before trusting a model.

In [ ]:
preview_count = min(9, len(records))
sample_records = records.sample(preview_count, random_state=RANDOM_STATE).reset_index(drop=True)
fig, axes = plt.subplots(3, 3, figsize=(12, 10))
axes = axes.flatten()

for ax in axes:
    ax.axis("off")

for ax, row in zip(axes, sample_records.itertuples(index=False)):
    with Image.open(row.filepath) as image:
        ax.imshow(image.convert("RGB"))
    label = getattr(row, target_column)
    ax.set_title(f"{label}\n{Path(row.relative_path).name}", fontsize=9)
    ax.axis("off")

plt.suptitle(f"Sample images for target: {TARGET_MODE}", fontsize=14)
plt.tight_layout()
plt.show()

## 6. Preprocess images for the classical baseline

We will resize every image to the same shape and convert it into a numerical array.

Why do this?
- Machine-learning models need consistent input shapes.
- A simple baseline helps us understand the workflow before using deep learning.


In [ ]:
started = time.perf_counter()
image_arrays, filtered_records = load_images_as_arrays(
    records,
    image_size=IMAGE_SIZE,
    color_mode="rgb",
    progress_title="Preprocess",
)
preprocess_elapsed = record_timing(step_timings, "preprocess_images", started)

# Extract the class label for each image.
labels = filtered_records[target_column].astype(str).to_numpy()
# Flatten each RGB image into one long feature vector and scale pixel values into
# the 0-to-1 range, which makes classical models easier to train.
X = image_arrays.reshape(len(image_arrays), -1) / 255.0

# These placeholders are used only for the later defect challenge section.
defect_challenge_X = None
defect_challenge_gray_records = None
if TARGET_MODE == "device_type" and RUN_DEFECT_CHALLENGE and not defect_challenge_records.empty:
    defect_arrays, defect_challenge_gray_records = load_images_as_arrays(
        defect_challenge_records,
        image_size=IMAGE_SIZE,
        color_mode="rgb",
        progress_title="Defect challenge",
    )
    defect_challenge_X = defect_arrays.reshape(len(defect_arrays), -1) / 255.0

print(f"Processed {len(filtered_records)} images in {preprocess_elapsed:.2f} seconds.")
print("Feature matrix shape:", X.shape)
print("Image tensor shape before flattening:", image_arrays.shape)
print(f"Active dataset root used for training: {DATASET_ROOT}")
if TARGET_MODE == "damage_status":
    training_damage_counts = filtered_records["damage_status"].value_counts(dropna=True).sort_index()
    print("Final damage-status counts used for training:")
    display(training_damage_counts.rename("count").to_frame())
if defect_challenge_X is not None:
    print(f"Prepared {len(defect_challenge_gray_records)} damaged challenge images for later device-type inference.")


## 7. Train/test split

Stop and predict before running:
- If the dataset is tiny, do you expect the score to be stable every time?
- Why might a different random split change the final accuracy?

In [ ]:
# Stratification tries to preserve the class balance in both the train and test sets.
stratify_labels = safe_stratify_labels(labels)
X_train, X_test, y_train, y_test, train_records, test_records = train_test_split(
    X,
    labels,
    filtered_records,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=stratify_labels,
)

print("Train set size:", len(X_train))
print("Test set size:", len(X_test))
print("Used stratified split:", stratify_labels is not None)

## 8. Shared evaluation helpers

These helper functions keep the evaluation format consistent across all models so students can compare them fairly.


In [ ]:
model_summaries = []


def make_scaled_pixel_estimator(classifier):
    steps = [("scale", StandardScaler())]
    if USE_PCA:
        steps.append(("pca", PCA(n_components=PCA_VARIANCE_TO_KEEP, random_state=RANDOM_STATE)))
    steps.append(("clf", classifier))
    return Pipeline(steps)


def _render_gallery_image(image_data, color_mode="grayscale", image_size=None):
    if color_mode == "grayscale":
        if image_size is None:
            raise ValueError("image_size is required for grayscale galleries.")
        return image_data.reshape(image_size, image_size), {"cmap": "gray"}
    if image_data.ndim == 1:
        if image_size is None:
            raise ValueError("image_size is required for flattened RGB galleries.")
        return image_data.reshape(image_size, image_size, 3), {}
    return image_data, {}


def _softmax_rows(values):
    values = np.asarray(values, dtype=float)
    shifted = values - values.max(axis=1, keepdims=True)
    exp_values = np.exp(shifted)
    return exp_values / exp_values.sum(axis=1, keepdims=True)


def estimate_prediction_confidence(estimator, X_values):
    if hasattr(estimator, "predict_proba"):
        probabilities = estimator.predict_proba(X_values)
        return probabilities.max(axis=1), "Confidence"

    if hasattr(estimator, "decision_function"):
        decision = estimator.decision_function(X_values)
        if np.ndim(decision) == 1:
            confidence = 1.0 / (1.0 + np.exp(-np.abs(decision)))
        else:
            confidence = _softmax_rows(decision).max(axis=1)
        return confidence, "Confidence"

    return None, None


def show_prediction_gallery(images, truths, preds, indices, title, color_mode="grayscale", image_size=None, max_images=6, confidences=None, confidence_label="Confidence"):
    selected = list(indices)[:max_images]
    if not selected:
        display(Markdown(f"**{title}:** none available in this split."))
        return

    columns = min(3, len(selected))
    rows = math.ceil(len(selected) / columns)
    fig, axes = plt.subplots(rows, columns, figsize=(4 * columns, 4 * rows))
    axes = np.atleast_1d(axes).flatten()

    for ax in axes:
        ax.axis("off")

    for ax, idx in zip(axes, selected):
        rendered, kwargs = _render_gallery_image(images[idx], color_mode=color_mode, image_size=image_size)
        ax.imshow(rendered, **kwargs)
        title_lines = [f"True: {truths[idx]}", f"Pred: {preds[idx]}"]
        if confidences is not None:
            title_lines.append(f"{confidence_label}: {float(confidences[idx]):.2f}")
        ax.set_title("\n".join(title_lines), fontsize=10)
        ax.axis("off")

    plt.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.show()


def evaluate_predictions(model_name, truths, preds, images, color_mode="grayscale", image_size=None, cmap="Blues", confidences=None, confidence_label="Confidence"):
    labels_sorted = sorted(pd.unique(np.array(truths, dtype=object)))
    cm = confusion_matrix(truths, preds, labels=labels_sorted)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap=cmap, xticklabels=labels_sorted, yticklabels=labels_sorted)
    plt.title(f"{model_name} confusion matrix")
    plt.xlabel("Predicted label")
    plt.ylabel("True label")
    plt.show()

    correct_indices = [idx for idx, (truth, pred) in enumerate(zip(truths, preds)) if truth == pred]
    incorrect_indices = [idx for idx, (truth, pred) in enumerate(zip(truths, preds)) if truth != pred]

    show_prediction_gallery(
        images,
        truths,
        preds,
        correct_indices,
        title=f"{model_name}: correctly predicted images",
        color_mode=color_mode,
        image_size=image_size,
        confidences=confidences,
        confidence_label=confidence_label,
    )
    show_prediction_gallery(
        images,
        truths,
        preds,
        incorrect_indices,
        title=f"{model_name}: incorrectly predicted images",
        color_mode=color_mode,
        image_size=image_size,
        confidences=confidences,
        confidence_label=confidence_label,
    )


def fit_and_evaluate_classical_model(model_name, estimator, X_train, y_train, X_test, y_test, image_arrays, image_size, timing_key, cmap="Blues", prediction_decoder=None):
    # Train the model on the training split.
    started = time.perf_counter()
    estimator.fit(X_train, y_train)
    train_elapsed = record_timing(step_timings, f"{timing_key}_train", started)

    estimator_uses_pca = hasattr(estimator, "named_steps") and "pca" in estimator.named_steps
    if estimator_uses_pca:
        pca_step = estimator.named_steps["pca"]
        print(f"{model_name} PCA components kept: {pca_step.n_components_} of {X_train.shape[1]} original features")

    # Ask the trained model to predict labels for the held-out test split.
    started = time.perf_counter()
    predictions = estimator.predict(X_test)
    predict_elapsed = record_timing(step_timings, f"{timing_key}_predict", started)

    # Compute a simple summary score plus optional confidence values when the model
    # exposes them.
    confidences, confidence_label = estimate_prediction_confidence(estimator, X_test)
    accuracy = accuracy_score(y_test, predictions)
    display_truths = prediction_decoder(np.asarray(y_test)) if prediction_decoder is not None else np.asarray(y_test)
    display_predictions = prediction_decoder(np.asarray(predictions)) if prediction_decoder is not None else np.asarray(predictions)
    print(f"{model_name} training time: {train_elapsed:.2f} seconds")
    print(f"{model_name} prediction time: {predict_elapsed:.4f} seconds")
    print(f"{model_name} accuracy: {accuracy:.3f}")
    if confidences is not None:
        print(f"Average {confidence_label.lower()}: {float(np.mean(confidences)):.3f}")
    print(classification_report(display_truths, display_predictions))

    # Reuse the same plots and error inspection tools for every classical model.
    evaluate_predictions(
        model_name,
        truths=list(display_truths),
        preds=list(display_predictions),
        images=image_arrays,
        color_mode="rgb",
        image_size=image_size,
        cmap=cmap,
        confidences=confidences,
        confidence_label=confidence_label or "Confidence",
    )

    # Save the detailed results so we can compare models later in one place.
    result = {
        "model": model_name,
        "accuracy": float(accuracy),
        "train_seconds": float(train_elapsed),
        "predict_seconds": float(predict_elapsed),
        "estimator": estimator,
        "predictions": display_predictions,
        "raw_predictions": predictions,
        "confidences": confidences,
        "confidence_label": confidence_label,
    }
    model_summaries.append({
        "model": model_name,
        "accuracy": round(float(accuracy), 3),
        "train_seconds": round(float(train_elapsed), 3),
        "predict_seconds": round(float(predict_elapsed), 4),
        "pca": "on" if estimator_uses_pca else "off",
    })
    return result


## 9. Logistic regression

This stays as the main linear classification baseline for the workshop.

Hyperparameter notes:
- `max_iter=2000`: gives the optimizer more chances to converge. If this is too small, training can stop before the model settles.
- `StandardScaler()`: not a hyperparameter itself, but it helps by putting pixel features on a comparable scale before logistic regression learns.


In [ ]:
logistic_model = make_scaled_pixel_estimator(
    LogisticRegression(max_iter=2000),
)
logistic_results = fit_and_evaluate_classical_model(
    model_name="Logistic Regression",
    estimator=logistic_model,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    image_arrays=X_test,
    image_size=IMAGE_SIZE,
    timing_key="logistic",
    cmap="Blues",
)


## 10. Decision tree

Decision trees are easier to explain visually, but they can also overfit tiny datasets quickly.

Hyperparameter notes:
- `random_state=RANDOM_STATE`: makes the learned tree reproducible.
- This notebook is using the default tree depth, so the tree can keep growing until the stopping rules are met. That can make it flexible, but also easier to overfit.


In [ ]:
decision_tree_model = DecisionTreeClassifier(random_state=RANDOM_STATE)
decision_tree_results = fit_and_evaluate_classical_model(
    model_name="Decision Tree",
    estimator=decision_tree_model,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    image_arrays=X_test,
    image_size=IMAGE_SIZE,
    timing_key="decision_tree",
    cmap="Oranges",
)


## 11. K-nearest neighbors

K-nearest neighbors compares each test image to the training set instead of learning a single decision boundary.

Hyperparameter notes:
- `n_neighbors=3`: each prediction is based on the 3 closest training examples. Smaller values react more to local detail; larger values make the boundary smoother.
- `StandardScaler()`: helps distance-based methods like KNN because feature scale affects which points count as nearest neighbors.


In [ ]:
knn_model = make_scaled_pixel_estimator(
    KNeighborsClassifier(n_neighbors=3),
)
knn_results = fit_and_evaluate_classical_model(
    model_name="K-Nearest Neighbors",
    estimator=knn_model,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    image_arrays=X_test,
    image_size=IMAGE_SIZE,
    timing_key="knn",
    cmap="Purples",
)


## 12. Linear SVC

Linear SVC is another linear classifier. It predicts labels directly, but it does not provide class probabilities by default.

Hyperparameter notes:
- `max_iter=5000`: allows more optimization steps before the solver stops.
- `random_state=RANDOM_STATE`: keeps the run reproducible.
- `StandardScaler()`: helps linear margin-based models train more stably on pixel features.


In [ ]:
linear_svc_model = make_scaled_pixel_estimator(
    LinearSVC(max_iter=5000, random_state=RANDOM_STATE),
)
linear_svc_results = fit_and_evaluate_classical_model(
    model_name="Linear SVC",
    estimator=linear_svc_model,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    image_arrays=X_test,
    image_size=IMAGE_SIZE,
    timing_key="linear_svc",
    cmap="Reds",
)


## 13. Multi-layer perceptron (MLP)

An MLP is still working on flattened RGB pixels like the other classical models, but it can learn non-linear relationships through hidden layers.

Teaching note: this gives us a neural-network baseline before the CNN section. The key difference is that the MLP still sees the image as one long feature vector, while the CNN keeps the 2D image layout and color channels.

Hyperparameter notes:
- `hidden_layer_sizes=(128,)`: uses one hidden layer with 128 neurons, which keeps the model expressive without becoming too large for this small dataset.
- `early_stopping=True`: automatically holds out part of the training data and stops when validation performance stops improving.
- `max_iter=500`: gives the optimizer enough passes to learn if early stopping does not trigger first.
- `StandardScaler()`: helps gradient-based neural networks train more reliably because the pixel features are on a comparable scale.


In [ ]:
mlp_results = None
mlp_label_encoder = None

try:
    # Encode the class labels as integers so sklearn's internal early-stopping
    # validation loop can score the MLP reliably on this environment.
    mlp_label_encoder = LabelEncoder()
    y_train_mlp = mlp_label_encoder.fit_transform(np.asarray(y_train))
    y_test_mlp = mlp_label_encoder.transform(np.asarray(y_test))

    # The MLP is a neural network, but it still works on flattened pixels.
    mlp_model = make_scaled_pixel_estimator(
        MLPClassifier(hidden_layer_sizes=(128,), early_stopping=True, max_iter=500, random_state=RANDOM_STATE),
    )
    mlp_results = fit_and_evaluate_classical_model(
        model_name="MLP",
        estimator=mlp_model,
        X_train=X_train,
        y_train=y_train_mlp,
        X_test=X_test,
        y_test=y_test_mlp,
        image_arrays=X_test,
        image_size=IMAGE_SIZE,
        timing_key="mlp",
        cmap="Greens",
        prediction_decoder=mlp_label_encoder.inverse_transform,
    )
except Exception as error:
    print("MLP skipped because training or evaluation failed.")
    print(f"MLP error: {type(error).__name__}: {error}")


## 14. Tiny CNN

Convolutional neural networks can learn spatial image patterns better than flattened classical baselines.

Fair-comparison note: in this notebook, every model uses RGB images so the comparison focuses on model behavior rather than whether color information was available.

CNN hyperparameter notes:
- `CNN_IMAGE_SIZE=64`: every image is resized to `64 x 64` before entering the CNN.
- `CNN_EPOCHS=6`: the model sees the full training set 6 times. More epochs can improve learning, but too many can overfit.
- `CNN_BATCH_SIZE=8`: the optimizer updates after every 8 training images. Smaller batches are noisier; larger batches are steadier but use more memory.
- `CNN_LEARNING_RATE=0.001`: controls how large each Adam update is. Too high can make training unstable; too low can make learning very slow.

CNN architecture notes:
- `Conv2d(..., 16, kernel_size=3, padding=1)`: first convolution layer learns 16 feature maps from `3x3` image patches while preserving image size.
- `MaxPool2d(2)`: halves the spatial width and height, which reduces computation and keeps stronger signals.
- `Conv2d(16, 32, ...)`: second convolution layer learns 32 higher-level feature maps.
- `Linear(..., 64)`: dense hidden layer with 64 units after the convolution blocks.
- `Dropout(0.2)`: randomly turns off 20% of hidden units during training to reduce overfitting.
- `Linear(64, num_classes)`: final output layer, one score per class.


In [ ]:
cnn_results = None
cnn_model = None
cnn_label_names = None
cnn_defect_challenge_records = None
# These variables let us optionally test the CNN on damaged images that were never
# part of the normal training or test split.
cnn_defect_challenge_arrays = None

def _prepare_cnn_challenge_arrays(records):
    # Prepare challenge images in the same format used by the CNN training pipeline.
    arrays, filtered = load_images_as_arrays(
        records,
        image_size=CNN_IMAGE_SIZE,
        color_mode="rgb",
        progress_title="CNN defect challenge",
    )
    return arrays, filtered

if not TORCH_AVAILABLE:
    display(Markdown("**CNN skipped:** install PyTorch from `requirements.txt` to run this section."))
else:
    # Load the RGB images for the CNN. Unlike the classical models, the CNN keeps
    # the 2D image layout and color channels instead of flattening immediately.
    started = time.perf_counter()
    cnn_arrays, cnn_records = load_images_as_arrays(
        filtered_records,
        image_size=CNN_IMAGE_SIZE,
        color_mode="rgb",
        progress_title="CNN images",
    )
    cnn_prep_elapsed = record_timing(step_timings, "cnn_preprocess", started)

    # Convert human-readable class names into integer IDs because neural networks
    # train on numbers, not strings.
    cnn_labels = cnn_records[target_column].astype(str).to_numpy()
    cnn_label_names = sorted(pd.unique(cnn_labels))
    label_to_index = {label: idx for idx, label in enumerate(cnn_label_names)}
    y_encoded = np.array([label_to_index[label] for label in cnn_labels])

    stratify_cnn = safe_stratify_labels(y_encoded)
    X_train_cnn, X_test_cnn, y_train_cnn, y_test_cnn = train_test_split(
        cnn_arrays,
        y_encoded,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        stratify=stratify_cnn,
    )

    # Rearrange the arrays from [batch, height, width, channels] into the PyTorch
    # format [batch, channels, height, width].
    X_train_tensor = torch.tensor(X_train_cnn.transpose(0, 3, 1, 2) / 255.0, dtype=torch.float32)
    X_test_tensor = torch.tensor(X_test_cnn.transpose(0, 3, 1, 2) / 255.0, dtype=torch.float32)
    y_train_tensor = torch.tensor(y_train_cnn, dtype=torch.long)
    y_test_tensor = torch.tensor(y_test_cnn, dtype=torch.long)

    # DataLoaders feed the network mini-batches instead of the entire dataset at once.
    train_loader = DataLoader(TensorDataset(X_train_tensor, y_train_tensor), batch_size=CNN_BATCH_SIZE, shuffle=True)
    test_loader = DataLoader(TensorDataset(X_test_tensor, y_test_tensor), batch_size=CNN_BATCH_SIZE, shuffle=False)

    # Define a very small CNN so students can see the full workflow without long run times.
    class TinyCNN(nn.Module):
        def __init__(self, num_classes):
            super().__init__()
            self.features = nn.Sequential(
                nn.Conv2d(3, 16, kernel_size=3, padding=1),
                nn.ReLU(),
                nn.MaxPool2d(2),
                nn.Conv2d(16, 32, kernel_size=3, padding=1),
                nn.ReLU(),
                nn.MaxPool2d(2),
            )
            self.classifier = nn.Sequential(
                nn.Flatten(),
                nn.Linear(32 * (CNN_IMAGE_SIZE // 4) * (CNN_IMAGE_SIZE // 4), 64),
                nn.ReLU(),
                nn.Dropout(0.2),
                nn.Linear(64, num_classes),
            )

        def forward(self, x):
            x = self.features(x)
            return self.classifier(x)

    # Cross-entropy is the standard loss for multi-class classification, and Adam is a
    # popular optimizer that usually works well out of the box.
    cnn_model = TinyCNN(num_classes=len(cnn_label_names))
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(cnn_model.parameters(), lr=CNN_LEARNING_RATE)

    epoch_bar, epoch_status = create_progress("CNN epochs", CNN_EPOCHS)
    started = time.perf_counter()
    training_history = []

    for epoch in range(CNN_EPOCHS):
        # Training mode turns on behaviors such as dropout.
        cnn_model.train()
        epoch_loss = 0.0
        for batch_inputs, batch_targets in train_loader:
            # Clear old gradients, run a forward pass, measure the error, then
            # backpropagate so the optimizer can update the weights.
            optimizer.zero_grad()
            outputs = cnn_model(batch_inputs)
            loss = criterion(outputs, batch_targets)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * len(batch_inputs)

        avg_loss = epoch_loss / len(train_loader.dataset)
        training_history.append(avg_loss)
        update_progress(
            epoch_bar,
            epoch_status,
            epoch + 1,
            f"Epoch {epoch + 1}/{CNN_EPOCHS} complete | average loss = {avg_loss:.4f}",
            done=epoch + 1 == CNN_EPOCHS,
        )

    cnn_train_elapsed = record_timing(step_timings, "cnn_train", started)

    # Switch to evaluation mode and collect predictions on the held-out test set.
    started = time.perf_counter()
    cnn_model.eval()
    cnn_pred_indices = []
    cnn_truth_indices = []
    with torch.no_grad():
        for batch_inputs, batch_targets in test_loader:
            logits = cnn_model(batch_inputs)
            cnn_pred_indices.extend(torch.argmax(logits, dim=1).cpu().numpy())
            cnn_truth_indices.extend(batch_targets.cpu().numpy())
    cnn_predict_elapsed = record_timing(step_timings, "cnn_predict", started)

    cnn_pred_indices = np.array(cnn_pred_indices)
    cnn_truth_indices = np.array(cnn_truth_indices)
    # Convert integer predictions back into the original class names for readable reports.
    decoded_preds = [cnn_label_names[idx] for idx in cnn_pred_indices]
    decoded_truths = [cnn_label_names[idx] for idx in cnn_truth_indices]
    cnn_accuracy = accuracy_score(decoded_truths, decoded_preds)

    print(f"CNN preprocessing time: {cnn_prep_elapsed:.2f} seconds")
    print(f"CNN training time: {cnn_train_elapsed:.2f} seconds")
    print(f"CNN prediction time: {cnn_predict_elapsed:.4f} seconds")
    print(f"CNN accuracy: {cnn_accuracy:.3f}")
    print(classification_report(decoded_truths, decoded_preds))

    plt.figure(figsize=(6, 4))
    plt.plot(range(1, CNN_EPOCHS + 1), training_history, marker="o")
    plt.title("CNN training loss by epoch")
    plt.xlabel("Epoch")
    plt.ylabel("Average loss")
    plt.show()

    cnn_confidences = torch.softmax(torch.tensor(np.column_stack([np.zeros(len(cnn_pred_indices)), np.zeros(len(cnn_pred_indices))])), dim=1) if False else None
    with torch.no_grad():
        full_test_logits = cnn_model(X_test_tensor)
        full_test_probs = torch.softmax(full_test_logits, dim=1).cpu().numpy()
    cnn_confidences = full_test_probs.max(axis=1)
    evaluate_predictions(
        model_name="Tiny CNN",
        truths=decoded_truths,
        preds=decoded_preds,
        images=X_test_cnn,
        color_mode="rgb",
        image_size=None,
        cmap="Greens",
        confidences=cnn_confidences,
        confidence_label="Confidence",
    )

    cnn_results = {
        "model": "Tiny CNN",
        "accuracy": float(cnn_accuracy),
        "train_seconds": float(cnn_train_elapsed),
        "predict_seconds": float(cnn_predict_elapsed),
    }
    model_summaries.append({
        "model": "Tiny CNN",
        "accuracy": round(float(cnn_accuracy), 3),
        "train_seconds": round(float(cnn_train_elapsed), 3),
        "predict_seconds": round(float(cnn_predict_elapsed), 4),
    })

    if TARGET_MODE == "device_type" and RUN_DEFECT_CHALLENGE and not defect_challenge_records.empty:
        cnn_defect_challenge_arrays, cnn_defect_challenge_records = _prepare_cnn_challenge_arrays(defect_challenge_records)


## 15. Defect challenge set: extrapolation check

This section is only meaningful in `device_type` mode. These `Defect` images were not used for training or for the standard test split.


In [ ]:
defect_challenge_prediction_table = pd.DataFrame()
defect_challenge_comparison_table = pd.DataFrame()

print(f"Defect challenge status: TARGET_MODE={TARGET_MODE}, RUN_DEFECT_CHALLENGE={RUN_DEFECT_CHALLENGE}")

if TARGET_MODE != "device_type":
    display(Markdown("**Defect challenge skipped:** this section only runs for `device_type` mode."))
elif not RUN_DEFECT_CHALLENGE or defect_challenge_X is None or defect_challenge_gray_records is None or defect_challenge_gray_records.empty:
    display(Markdown("**Defect challenge skipped:** no damaged challenge images were prepared."))
else:
    defect_truths = defect_challenge_gray_records["device_type"].astype(str).tolist()
    challenge_rows = []
    challenge_model_outputs = []
    challenge_prediction_table = defect_challenge_gray_records[["relative_path", "device_type"]].copy()
    challenge_prediction_table = challenge_prediction_table.rename(columns={"device_type": "true_device_type"})

    classical_challenge_models = [
        ("Logistic Regression", logistic_results, None),
        ("Decision Tree", decision_tree_results, None),
        ("K-Nearest Neighbors", knn_results, None),
        ("Linear SVC", linear_svc_results, None),
    ]

    if "mlp_results" in globals() and mlp_results is not None and mlp_label_encoder is not None:
        classical_challenge_models.append(("MLP", mlp_results, mlp_label_encoder.inverse_transform))
    else:
        print("MLP skipped in defect challenge because no trained MLP result is available.")

    for model_name, model_result, prediction_decoder in classical_challenge_models:
        estimator = model_result["estimator"]
        raw_predictions = estimator.predict(defect_challenge_X)
        predictions = prediction_decoder(raw_predictions) if prediction_decoder is not None else raw_predictions
        predictions = np.asarray(predictions, dtype=str)
        confidences, confidence_label = estimate_prediction_confidence(estimator, defect_challenge_X)
        challenge_accuracy = accuracy_score(defect_truths, predictions)
        avg_confidence = float(np.mean(confidences)) if confidences is not None else np.nan

        column_stub = model_name.lower().replace(" ", "_").replace("-", "")
        challenge_prediction_table[f"{column_stub}_predicted_device"] = predictions
        challenge_prediction_table[f"{column_stub}_correct"] = challenge_prediction_table["true_device_type"] == predictions
        if confidences is not None:
            challenge_prediction_table[f"{column_stub}_confidence"] = np.round(confidences, 3)

        challenge_rows.append({
            "model": model_name,
            "defect_challenge_accuracy": round(float(challenge_accuracy), 3),
            "avg_confidence": round(avg_confidence, 3) if not np.isnan(avg_confidence) else np.nan,
            "challenge_images": len(defect_truths),
        })
        challenge_model_outputs.append({
            "model": model_name,
            "predictions": predictions,
            "confidences": confidences,
        })
        print(f"{model_name} defect challenge accuracy: {challenge_accuracy:.3f}")

    if cnn_model is not None and cnn_defect_challenge_arrays is not None and cnn_defect_challenge_records is not None and not cnn_defect_challenge_records.empty:
        cnn_inputs = torch.tensor(cnn_defect_challenge_arrays.transpose(0, 3, 1, 2) / 255.0, dtype=torch.float32)
        cnn_model.eval()
        with torch.no_grad():
            cnn_logits = cnn_model(cnn_inputs)
            cnn_probs = torch.softmax(cnn_logits, dim=1).cpu().numpy()
        cnn_pred_idx = cnn_probs.argmax(axis=1)
        cnn_pred_labels = [cnn_label_names[idx] for idx in cnn_pred_idx]
        cnn_confidence = cnn_probs.max(axis=1)
        cnn_challenge_truths = cnn_defect_challenge_records["device_type"].astype(str).tolist()
        cnn_challenge_accuracy = accuracy_score(cnn_challenge_truths, cnn_pred_labels)

        challenge_prediction_table["tiny_cnn_predicted_device"] = cnn_pred_labels
        challenge_prediction_table["tiny_cnn_correct"] = challenge_prediction_table["true_device_type"] == cnn_pred_labels
        challenge_prediction_table["tiny_cnn_confidence"] = np.round(cnn_confidence, 3)
        challenge_rows.append({
            "model": "Tiny CNN",
            "defect_challenge_accuracy": round(float(cnn_challenge_accuracy), 3),
            "avg_confidence": round(float(np.mean(cnn_confidence)), 3),
            "challenge_images": len(cnn_challenge_truths),
        })
        challenge_model_outputs.append({
            "model": "Tiny CNN",
            "predictions": np.asarray(cnn_pred_labels, dtype=str),
            "confidences": cnn_confidence,
        })
        print(f"Tiny CNN defect challenge accuracy: {cnn_challenge_accuracy:.3f}")

    defect_challenge_prediction_table = challenge_prediction_table.copy()
    display(defect_challenge_prediction_table.head(10))

    defect_challenge_comparison_table = pd.DataFrame(challenge_rows)
    defect_challenge_comparison_table = defect_challenge_comparison_table.sort_values(
        by="defect_challenge_accuracy",
        ascending=False,
    ).reset_index(drop=True)
    display(defect_challenge_comparison_table)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    defect_challenge_comparison_table.plot(
        kind="bar",
        x="model",
        y="defect_challenge_accuracy",
        color="#4c78a8",
        legend=False,
        ax=axes[0],
        title="Defect challenge accuracy by model",
    )
    axes[0].set_ylabel("Accuracy")
    axes[0].set_ylim(0, 1)
    axes[0].tick_params(axis="x", rotation=20)

    defect_challenge_comparison_table.plot(
        kind="bar",
        x="model",
        y="avg_confidence",
        color="#f58518",
        legend=False,
        ax=axes[1],
        title="Average confidence on defect images",
    )
    axes[1].set_ylabel("Average confidence")
    axes[1].set_ylim(0, 1)
    axes[1].tick_params(axis="x", rotation=20)

    plt.tight_layout()
    plt.show()

    if challenge_model_outputs:
        challenge_labels = sorted(pd.unique(np.asarray(defect_truths, dtype=object)))
        matrix_count = len(challenge_model_outputs)
        matrix_cols = min(3, matrix_count)
        matrix_rows = math.ceil(matrix_count / matrix_cols)
        fig, axes = plt.subplots(matrix_rows, matrix_cols, figsize=(4.5 * matrix_cols, 4 * matrix_rows))
        axes = np.atleast_1d(axes).flatten()

        for ax in axes:
            ax.axis("off")

        for ax, output in zip(axes, challenge_model_outputs):
            cm = confusion_matrix(defect_truths, output["predictions"], labels=challenge_labels)
            sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=challenge_labels, yticklabels=challenge_labels, ax=ax)
            ax.set_title(f"{output['model']} defect challenge")
            ax.set_xlabel("Predicted device")
            ax.set_ylabel("True device")

        plt.tight_layout()
        plt.show()

        gallery_count = min(8, len(defect_truths))
        gallery_cols = min(2, gallery_count)
        gallery_rows = math.ceil(gallery_count / gallery_cols)
        fig, axes = plt.subplots(gallery_rows, gallery_cols, figsize=(6 * gallery_cols, 4.5 * gallery_rows))
        axes = np.atleast_1d(axes).flatten()

        for ax in axes:
            ax.axis("off")

        for ax, idx in zip(axes, range(gallery_count)):
            rendered, kwargs = _render_gallery_image(defect_challenge_X[idx], color_mode="rgb", image_size=IMAGE_SIZE)
            ax.imshow(rendered, **kwargs)
            title_lines = [f"True: {defect_truths[idx]}"]
            for output in challenge_model_outputs:
                pred = output["predictions"][idx]
                status = "OK" if pred == defect_truths[idx] else "MISS"
                title_lines.append(f"{output['model']}: {pred} ({status})")
            ax.set_title("\n".join(title_lines), fontsize=8)
            ax.axis("off")

        plt.suptitle("Defect challenge image-level model decisions", fontsize=14)
        plt.tight_layout()
        plt.show()


## 16. Final comparison summary

This section compares the standard clean-test results across all available methods so students can see the tradeoffs in one place.


In [ ]:
runtime_table = timing_frame(step_timings)
display(runtime_table)

comparison_table = pd.DataFrame(model_summaries)
comparison_table = comparison_table.sort_values(by="accuracy", ascending=False).reset_index(drop=True)

display(Markdown("### Clean test split comparison"))
display(comparison_table)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
comparison_table.plot(kind="bar", x="model", y="accuracy", color="#4c78a8", legend=False, ax=axes[0], title="Clean test accuracy by model")
axes[0].set_ylabel("Accuracy")
axes[0].set_ylim(0, 1)
axes[0].tick_params(axis="x", rotation=20)

comparison_table.plot(kind="bar", x="model", y="train_seconds", color="#f58518", legend=False, ax=axes[1], title="Training time by model")
axes[1].set_ylabel("Seconds")
axes[1].tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.show()

if "defect_challenge_comparison_table" in globals() and not defect_challenge_comparison_table.empty:
    display(Markdown("### Held-out defect challenge comparison"))
    display(defect_challenge_comparison_table)
else:
    display(Markdown(
        "**Held-out defect challenge comparison unavailable:** run the defect challenge section in `device_type` mode first."
    ))

display(Markdown(
    "**Reflection prompts:** Which model was most accurate on the clean split? "
    "Did the ranking change on the held-out defect images? Which models were confident but wrong?"
))


## 17. What students should notice

The dataset is small .

Reflect on these questions after running the notebook:
- Which model performed best on the clean test set?
- Which model trained fastest?
- Did the more complex model always win?
- How did the models behave on damaged images compared with the clean test split?
- Would you trust high confidence on the defect challenge set if there is no true label to compare against?
- Could augmentation improve fairness across classes?

Next step:
- use the augmentation notebook to generate a larger training set without editing the original image folders.


## 18. Boundary visual aid

These plots compress the image feature space into two PCA coordinates and show each model's predicted regions on that 2D map. This is a teaching visualization, not the full high-dimensional boundary.

In [ ]:
BOUNDARY_GRID_RESOLUTION = 90  # Student change point: lower is faster; higher is smoother.
BOUNDARY_CNN_BATCH_SIZE = 512
BOUNDARY_COLOR_MODE = "rgb"
BOUNDARY_TOOLTIP_THUMBNAIL_SIZE = 120
BOUNDARY_TOOLTIP_MAX_POINTS = 700  # Student change point: lower this for faster notebook rendering.

boundary_model_specs = [
    ("Logistic Regression", logistic_results, None),
    ("Decision Tree", decision_tree_results, None),
    ("K-Nearest Neighbors", knn_results, None),
    ("Linear SVC", linear_svc_results, None),
]

if "mlp_results" in globals() and mlp_results is not None and mlp_label_encoder is not None:
    boundary_model_specs.append(("MLP", mlp_results, mlp_label_encoder.inverse_transform))
else:
    print("MLP boundary skipped because no trained MLP result is available.")

def predict_cnn_on_pca_reconstructions(flat_values):
    if not TORCH_AVAILABLE or cnn_model is None or cnn_label_names is None:
        return None

    images = flat_values.reshape(-1, IMAGE_SIZE, IMAGE_SIZE, 3)
    if CNN_IMAGE_SIZE != IMAGE_SIZE:
        resized_images = []
        for image_array in images:
            pil_image = Image.fromarray(np.clip(image_array * 255, 0, 255).astype(np.uint8), mode="RGB")
            pil_image = pil_image.resize((CNN_IMAGE_SIZE, CNN_IMAGE_SIZE))
            resized_images.append(np.array(pil_image, dtype=np.float32) / 255.0)
        images = np.stack(resized_images)

    cnn_inputs = torch.tensor(images.transpose(0, 3, 1, 2), dtype=torch.float32)
    pred_indices = []
    cnn_model.eval()
    with torch.no_grad():
        for start in range(0, len(cnn_inputs), BOUNDARY_CNN_BATCH_SIZE):
            logits = cnn_model(cnn_inputs[start:start + BOUNDARY_CNN_BATCH_SIZE])
            pred_indices.extend(torch.argmax(logits, dim=1).cpu().numpy())

    return np.asarray([cnn_label_names[index] for index in pred_indices], dtype=str)

def make_boundary_thumbnail_source(records_for_points, coordinates, labels_for_points, split_name, label_to_int, cmap):
    rows = []
    for row, coordinate, label in zip(records_for_points.itertuples(index=False), coordinates, labels_for_points):
        try:
            with Image.open(row.filepath) as image:
                thumbnail = image.convert("RGB")
                thumbnail.thumbnail((BOUNDARY_TOOLTIP_THUMBNAIL_SIZE, BOUNDARY_TOOLTIP_THUMBNAIL_SIZE))
                buffer = BytesIO()
                thumbnail.save(buffer, format="JPEG", quality=78)
                thumbnail_source = "data:image/jpeg;base64," + base64.b64encode(buffer.getvalue()).decode("ascii")
        except Exception:
            thumbnail_source = ""

        rgba = cmap(label_to_int[str(label)])
        rows.append(
            {
                "pc1": float(coordinate[0]),
                "pc2": float(coordinate[1]),
                "label": str(label),
                "split": split_name,
                "image_name": str(row.relative_path),
                "thumbnail": thumbnail_source,
                "color": f"rgb({int(rgba[0] * 255)}, {int(rgba[1] * 255)}, {int(rgba[2] * 255)})",
            }
        )
    return ColumnDataSource(pd.DataFrame(rows))

def grid_predictions_to_rgba(grid_int, cmap):
    rgba_bytes = np.empty((*grid_int.shape, 4), dtype=np.uint8)
    for class_index in range(cmap.N):
        mask = grid_int == class_index
        color = cmap(class_index)
        rgba_bytes[mask, 0] = int(color[0] * 255)
        rgba_bytes[mask, 1] = int(color[1] * 255)
        rgba_bytes[mask, 2] = int(color[2] * 255)
        rgba_bytes[mask, 3] = 64
    return np.ascontiguousarray(rgba_bytes).view(dtype=np.uint32).reshape(grid_int.shape)

if X_train.shape[0] < 2:
    display(Markdown("**Boundary plot skipped:** PCA needs at least two training images."))
else:
    pca_boundary = PCA(n_components=2, random_state=RANDOM_STATE)
    train_2d = pca_boundary.fit_transform(X_train)
    test_2d = pca_boundary.transform(X_test)

    x_min, x_max = train_2d[:, 0].min(), train_2d[:, 0].max()
    y_min, y_max = train_2d[:, 1].min(), train_2d[:, 1].max()
    x_pad = max((x_max - x_min) * 0.15, 0.1)
    y_pad = max((y_max - y_min) * 0.15, 0.1)
    xx, yy = np.meshgrid(
        np.linspace(x_min - x_pad, x_max + x_pad, BOUNDARY_GRID_RESOLUTION),
        np.linspace(y_min - y_pad, y_max + y_pad, BOUNDARY_GRID_RESOLUTION),
    )
    grid_2d = np.c_[xx.ravel(), yy.ravel()]
    grid_original_space = np.clip(pca_boundary.inverse_transform(grid_2d), 0, 1)

    boundary_outputs = []
    for model_name, model_result, prediction_decoder in boundary_model_specs:
        estimator = model_result["estimator"]
        raw_predictions = estimator.predict(grid_original_space)
        predictions = prediction_decoder(raw_predictions) if prediction_decoder is not None else raw_predictions
        boundary_outputs.append({"model": model_name, "grid_predictions": np.asarray(predictions, dtype=str)})

    cnn_grid_predictions = predict_cnn_on_pca_reconstructions(grid_original_space)
    if cnn_grid_predictions is not None:
        boundary_outputs.append({"model": "Tiny CNN", "grid_predictions": cnn_grid_predictions})
    else:
        print("Tiny CNN boundary skipped because no trained CNN is available.")

    train_labels = np.asarray(y_train, dtype=str)
    test_labels = np.asarray(y_test, dtype=str)
    point_label_values = list(train_labels) + list(test_labels)
    if defect_challenge_X is not None and defect_challenge_gray_records is not None and not defect_challenge_gray_records.empty:
        defect_2d = pca_boundary.transform(defect_challenge_X)
        defect_labels = defect_challenge_gray_records["device_type"].astype(str).to_numpy()
        point_label_values.extend(defect_labels.tolist())
    else:
        defect_2d = None
        defect_labels = np.array([])

    label_values = point_label_values.copy()
    for output in boundary_outputs:
        label_values.extend(output["grid_predictions"].tolist())

    labels_sorted = sorted(pd.unique(np.asarray(label_values, dtype=object)))
    point_labels_sorted = sorted(pd.unique(np.asarray(point_label_values, dtype=object)))
    label_to_int = {label: index for index, label in enumerate(labels_sorted)}
    cmap = plt.get_cmap("tab10", max(len(labels_sorted), 1))

    tooltip_point_count = len(train_records) + len(test_records)
    if defect_2d is not None:
        tooltip_point_count += len(defect_challenge_gray_records)

    if tooltip_point_count > BOUNDARY_TOOLTIP_MAX_POINTS:
        display(Markdown(
            f"**Interactive image tooltip plot skipped:** `{tooltip_point_count}` images would make the notebook output heavy. "
            f"Raise `BOUNDARY_TOOLTIP_MAX_POINTS` or lower the dataset size if you want thumbnail hovers."
        ))
    else:
        try:
            import base64
            from io import BytesIO
            from bokeh.io import output_notebook, show
            from bokeh.layouts import gridplot
            from bokeh.models import ColumnDataSource, HoverTool, Span
            from bokeh.plotting import figure

            output_notebook()
            tooltip_html = """
                <div style="width: 170px;">
                    <div style="font-weight: 700; margin-bottom: 4px;">@image_name</div>
                    <div style="font-size: 12px; color: #555;">Label: @label</div>
                    <div style="font-size: 12px; margin-bottom: 6px; color: #555;">Split: @split</div>
                    <img src="@thumbnail" style="width: 120px; max-height: 120px; object-fit: contain; border: 1px solid #ddd;" />
                    <div style="font-size: 12px; margin-top: 6px; color: #555;">PC1: @pc1{0.000}</div>
                    <div style="font-size: 12px; color: #555;">PC2: @pc2{0.000}</div>
                </div>
                """
            train_source = make_boundary_thumbnail_source(train_records, train_2d, train_labels, "train", label_to_int, cmap)
            test_source = make_boundary_thumbnail_source(test_records, test_2d, test_labels, "clean test", label_to_int, cmap)
            defect_source = None
            if defect_2d is not None:
                defect_source = make_boundary_thumbnail_source(defect_challenge_gray_records, defect_2d, defect_labels, "defect challenge", label_to_int, cmap)

            for output in boundary_outputs:
                grid_int = np.array([label_to_int[label] for label in output["grid_predictions"]]).reshape(xx.shape)
                output["grid_int"] = grid_int

            panel_count = len(boundary_outputs)
            columns = min(3, panel_count)
            x_start = float(xx.min())
            y_start = float(yy.min())
            x_span = float(xx.max() - xx.min())
            y_span = float(yy.max() - yy.min())
            bokeh_figures = []

            for output in boundary_outputs:
                hover_tool = HoverTool(tooltips=tooltip_html)
                boundary_figure = figure(
                    title=f"{output['model']} - PCA image boundary view",
                    width=420,
                    height=360,
                    tools=[hover_tool, "pan", "wheel_zoom", "box_zoom", "reset", "save"],
                    x_range=(x_start, x_start + x_span),
                    y_range=(y_start, y_start + y_span),
                    x_axis_label="PCA 1",
                    y_axis_label="PCA 2",
                )
                boundary_figure.image_rgba(
                    image=[grid_predictions_to_rgba(output["grid_int"], cmap)],
                    x=x_start,
                    y=y_start,
                    dw=x_span,
                    dh=y_span,
                )
                boundary_figure.add_layout(Span(location=0, dimension="width", line_color="gray", line_alpha=0.35))
                boundary_figure.add_layout(Span(location=0, dimension="height", line_color="gray", line_alpha=0.35))

                hover_renderers = []
                train_renderer = boundary_figure.scatter("pc1", "pc2", source=train_source, marker="circle", size=8, fill_color="color", line_color="white", line_width=0.5, alpha=0.88)
                test_renderer = boundary_figure.scatter("pc1", "pc2", source=test_source, marker="triangle", size=10, fill_color="color", line_color="black", line_width=0.6, alpha=0.95)
                hover_renderers.extend([train_renderer, test_renderer])
                if defect_source is not None:
                    defect_renderer = boundary_figure.scatter("pc1", "pc2", source=defect_source, marker="star", size=14, fill_color="color", line_color="black", line_width=0.8, alpha=1.0)
                    hover_renderers.append(defect_renderer)
                hover_tool.renderers = hover_renderers

                legend_x = x_start - x_span
                legend_y = y_start - y_span
                region_labels_sorted = sorted(pd.unique(output["grid_predictions"]))
                for label in region_labels_sorted:
                    rgba = cmap(label_to_int[label])
                    label_color = f"rgb({int(rgba[0] * 255)}, {int(rgba[1] * 255)}, {int(rgba[2] * 255)})"
                    boundary_figure.scatter([legend_x], [legend_y], marker="square", size=12, fill_color=label_color, fill_alpha=0.25, line_color=label_color, legend_label=f"region predicts: {label}")

                boundary_figure.scatter([legend_x], [legend_y], marker="circle", size=8, fill_color="lightgray", line_color="black", legend_label="dots: training images")
                boundary_figure.scatter([legend_x], [legend_y], marker="triangle", size=9, fill_color="lightgray", line_color="black", legend_label="triangles: clean test images")
                if defect_source is not None:
                    boundary_figure.scatter([legend_x], [legend_y], marker="star", size=12, fill_color="lightgray", line_color="black", legend_label="stars: defect challenge images")

                boundary_figure.legend.location = "top_right"
                boundary_figure.legend.title = "Legend"
                boundary_figure.legend.background_fill_alpha = 0.2
                boundary_figure.grid.grid_line_alpha = 0.25
                bokeh_figures.append(boundary_figure)

            show(gridplot(bokeh_figures, ncols=columns))
        except Exception as error:
            display(Markdown(f"**Interactive image tooltip plot skipped:** {error}"))

    display(Markdown(
        "**Boundary note:** each background region is produced by reconstructing PCA grid points back into pixel space, "
        "then asking the trained model for a prediction. CNN boundaries are approximate because the reconstructed PCA images are resized into the CNN input shape. "
        "Hover tooltips are attached only to image points, not to the background regions."
    ))